# Usar o Xselo no Colab (sem treinar)

Este notebook só **liga** o Xselo que você já treinou: pega o modelo do seu Google Drive e deixa ele pronto em três modos (escolha na célula 1):

- **`api`** (padrão): vira uma API igual à da OpenAI, com link público. No fim aparecem a **URL base**, a **API key** e o **modelo** pra colar em qualquer site ou app que aceite *OpenAI compatible* / *custom endpoint* / *reverse proxy* (SillyTavern, Open WebUI, Janitor AI...). Ligue o **streaming** no site.
- **`chat`**: conversa aqui mesmo, na caixinha `você>` (`/foto` manda imagem).
- **`discord`**: liga o bot do Discord (token em 🔑 Secrets como `DISCORD_TOKEN`).

**Antes:** *Ambiente de execução → Alterar o tipo de ambiente de execução → **A100***. Depois, **Executar tudo**.

Na A100 ele responde bem mais rápido que no Kaggle e aguenta conversas bem maiores (contexto de ~32 mil tokens contra ~8 mil). Mas aqui **gasta unidades de computação** do Colab Pro enquanto estiver ligado; o Kaggle é de graça. Quando terminar: *Ambiente de execução → Desconectar e excluir ambiente de execução*. E não feche a aba enquanto estiver usando: o Colab desliga sessões sem a aba aberta.

O link da API muda toda vez que liga. A API key fica fixa se você guardar uma em 🔑 Secrets como `XSELO_CHAVE`.

In [ ]:
#@title ⚙️ 1. Configuração
MODO = "api"  #@param ["api", "chat", "discord"]
SYSTEM_DO_SITE = "juntar"  #@param ["juntar", "substituir"]
BUSCA_NA_INTERNET = True  #@param {type:"boolean"}
PASTA_NO_DRIVE = "MyDrive/Ratex"  #@param {type:"string"}
ENVIAR_PRO_HUGGING_FACE = False  #@param {type:"boolean"}
REPO_PUBLICO = False  #@param {type:"boolean"}
# "juntar": o system prompt do site entra junto com o do Xselo; "substituir": o do site manda (bom pra personagens)
# O notebook procura o Xselo em PASTA_NO_DRIVE e usa o mais polido que achar (ex.: ...-polido-95pares).

In [ ]:
#@title 📦 2. Código do Ratex e bibliotecas (antes de qualquer coisa carregar o torch)
import os, subprocess, sys
if os.path.isdir("/content/Ratex/.git"):
    !git -C /content/Ratex pull -q
else:
    !git clone -q --depth 1 https://github.com/SeniorVortex/Ratex.git /content/Ratex
# o torch, o torchvision e o pillow do Colab NÃO são atualizados: já combinam entre si e com a GPU
!pip install -q -U "transformers>=5" "peft>=0.17" accelerate bitsandbytes safetensors huggingface_hub "discord.py>=2.4" fastapi uvicorn
!git -C /content/Ratex log --oneline -1

teste = subprocess.run([sys.executable, "-c",
    "import sys, torch, torchvision, transformers, peft; "
    "from transformers import AutoModelForImageTextToText, AutoProcessor; "
    "print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
    "'| transformers', transformers.__version__, '| peft', peft.__version__)"],
    capture_output=True, text=True)
if teste.returncode != 0:
    print(teste.stderr[-2500:])
    raise SystemExit("❌ As bibliotecas estão brigando entre si. Vá em Ambiente de execução → Desconectar e excluir "
                     "ambiente de execução, conecte de novo na A100 e rode tudo. Se continuar, mande o texto acima pro Claude.")
print("✅", teste.stdout.strip())

if "torch" in sys.modules:
    from importlib.metadata import version
    if version("torch").split("+")[0] != sys.modules["torch"].__version__.split("+")[0]:
        print("🔄 O torch foi atualizado com uma versão antiga ainda carregada. O Colab vai reiniciar sozinho:")
        print("   espere reconectar e clique de novo em Executar tudo. Da 2ª vez passa direto.")
        import time
        time.sleep(3)
        os.kill(os.getpid(), 9)

In [ ]:
#@title 💾 3. Achar o Xselo no seu Drive (ou mandar o .zip)
import os, sys, json, shutil, zipfile
os.chdir("/content/Ratex"); sys.path.insert(0, "/content/Ratex")
import torch
if not torch.cuda.is_available():
    raise SystemExit("Sem GPU! Ambiente de execução → Alterar o tipo de ambiente de execução → A100.")
VRAM = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {torch.cuda.get_device_name(0)} ({VRAM:.0f} GB)")
if VRAM < 22:
    raise SystemExit("O Xselo 31B precisa de uma GPU de 24 GB ou mais (A100 ou L4). Troque o tipo de ambiente.")
CONTEXTO = 32000 if VRAM >= 39 else 8000   # quanto de conversa cabe na memória que sobra

from nucleo.nuvem import achar_xselo
PASTA_XSELO = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_XSELO = achar_xselo(f"/content/drive/{PASTA_NO_DRIVE}")
except Exception as erro:
    print(f"⚠️ não deu pra usar o Drive ({erro})")
if PASTA_XSELO is None:
    print("Não achei o Xselo no Drive. Escolha o .zip do modelo (o que a célula 8 do notebook de treino baixou):")
    from google.colab import files
    enviados = files.upload()
    for nome in enviados:
        if nome.endswith(".zip"):
            with zipfile.ZipFile(nome) as z:
                z.extractall(f"/content/xselo_zip/{nome[:-4]}")
    PASTA_XSELO = achar_xselo("/content/xselo_zip")
if PASTA_XSELO is None:
    raise SystemExit("Não achei nenhum Xselo treinado (pasta com ratex_config.json e adapter_config.json).")
# copia pro disco do Colab: ler direto do Drive é lento
LOCAL = f"/content/xselo/{PASTA_XSELO.name}"
shutil.copytree(PASTA_XSELO, LOCAL, dirs_exist_ok=True)
print("✅ Xselo:", PASTA_XSELO.name, "| contexto da API:", CONTEXTO, "tokens")

In [ ]:
#@title 🤗 3b. (Opcional) Enviar o Xselo pro Hugging Face (marque ENVIAR_PRO_HUGGING_FACE na célula 1)
# Precisa de um token de ESCRITA do Hugging Face guardado na 🔑 (Secrets) com o nome HF_TOKEN.
if not ENVIAR_PRO_HUGGING_FACE:
    print("(envio pro Hugging Face desligado na célula 1)")
else:
    from google.colab import userdata
    from huggingface_hub import HfApi
    api = HfApi(token=userdata.get("HF_TOKEN"))
    usuario = api.whoami()["name"]
    REPO = f"{usuario}/{PASTA_XSELO.name}"
    api.create_repo(REPO, private=not REPO_PUBLICO, exist_ok=True)
    api.upload_folder(folder_path=LOCAL, repo_id=REPO, commit_message=f"Xselo: {PASTA_XSELO.name}",
                      ignore_patterns=["autopolimento.json", "*.log"])
    print(f"✅ enviado: https://huggingface.co/{REPO} ({'público' if REPO_PUBLICO else 'privado'})")

In [ ]:
#@title 🧠 4. Carregar o Xselo (na 1ª vez baixa a Gemma, uns 60 GB: ~5-10 min)
import time
from nucleo.hibrido import carregar_hibrido, enxerga
from nucleo.memoria import Memoria
t0 = time.time()
if "xselo" not in globals():
    xselo, tok, cfg = carregar_hibrido(LOCAL, device="cuda")
    memoria = Memoria()
print(f"✅ {cfg['nome']} carregado em {(time.time() - t0) / 60:.0f} min | enxerga imagens: {'sim' if enxerga(xselo) else 'não'} "
      f"| GPU: {torch.cuda.memory_allocated() / 2**30:.0f} GB ocupados")

In [ ]:
#@title 🌐 5. Busca na internet (SearXNG; na 1ª vez leva 1-2 min)
busca = None
if BUSCA_NA_INTERNET:
    try:
        from nucleo.busca import Busca, iniciar_searxng
        busca = Busca(iniciar_searxng())
        print("✅ busca funcionando. O Xselo pesquisa sozinho quando a pergunta é sobre coisa atual; pra forçar, comece com /buscar")
    except Exception as erro:
        busca = None
        print(f"⚠️ não deu pra ligar a busca ({erro}). Tudo bem: o Xselo responde sem ela.")

In [ ]:
#@title 💬 6. MODO = "chat": conversar aqui (/sair; /foto <pergunta> manda imagem; /lembrancas; /esquecer)
from nucleo.hibrido import responder
from nucleo.lembrancas import Conversa, Lembrancas, resumidor
ARQ_LEMBRANCAS = "/content/drive/MyDrive/Ratex/lembrancas.json" if os.path.isdir("/content/drive/MyDrive") else "/content/lembrancas.json"
conversa = Conversa("voce", Lembrancas(ARQ_LEMBRANCAS), resumir=resumidor(xselo, tok))
if MODO != "chat":
    print(f'(modo "{MODO}": pulando o chat)')
while MODO == "chat":
    msg = input("você> ").strip()
    if msg in ("/sair", "sair", ""):
        break
    aviso = conversa.comando(msg)
    if aviso:
        print(aviso)
        continue
    imagens = []
    if msg.startswith("/foto"):
        from google.colab import files
        imagens = list(files.upload())
        msg = msg[len("/foto"):].strip() or "O que você vê nessa imagem?"
    conversa.usuario(msg, imagens)
    print("xselo> ", end="")
    try:
        resposta = responder(xselo, tok, conversa.historico, system_prompt=conversa.system(cfg["system_prompt"]),
                             memoria=memoria, busca=busca, **cfg["geracao"])
    except Exception as erro:
        print(f"({erro})")
        conversa.historico.pop()
        continue
    conversa.xselo(resposta)
    print()

In [ ]:
#@title 🤖 7. MODO = "discord": bot do Discord (fica rodando até você parar a célula)
if MODO != "discord":
    print(f'(modo "{MODO}": bot do Discord desligado)')
else:
    from google.colab import userdata
    import bot_discord
    from nucleo.lembrancas import Lembrancas
    bot = bot_discord.XseloDiscord(xselo, tok, cfg, memoria=memoria, busca=busca,
                                   lembrancas=Lembrancas(ARQ_LEMBRANCAS.replace("lembrancas.json", "lembrancas_discord.json")))
    cliente = bot_discord.criar_cliente(bot)
    await cliente.start(userdata.get("DISCORD_TOKEN"))

In [ ]:
#@title 🔌 8. MODO = "api": API igual à da OpenAI com link público (fica rodando até você parar a célula)
import secrets, time
if MODO != "api":
    print(f'(modo "{MODO}": API desligada)')
else:
    from servidor_api import abrir_tunel, criar_app, ligar_servidor
    try:
        from google.colab import userdata
        CHAVE = userdata.get("XSELO_CHAVE")
    except Exception:
        CHAVE = "xselo-" + secrets.token_urlsafe(18)
    NOME_API = f"xselo-{cfg.get('versao', '')}"
    app = criar_app(xselo, tok, cfg, CHAVE, NOME_API, memoria=memoria, busca=busca, modo_system=SYSTEM_DO_SITE,
                    contexto=CONTEXTO)
    ligar_servidor(app, 8000)
    URL = abrir_tunel(8000)
    print("✅ Xselo no ar! Cole isso no site/app (campo OpenAI compatible / custom endpoint / reverse proxy):\n")
    print(f"   URL base:  {URL}/v1")
    print(f"   API key:   {CHAVE}")
    print(f"   Modelo:    {NOME_API}\n")
    print(f"No site: streaming ligado, contexto (Context Size) até {CONTEXTO}, resposta de 400 a 800 tokens.")
    print("Não mostre a API key pra ninguém que você não queira usando a sua GPU. Pra desligar: ⏹ nesta célula.")
    inicio = time.time()
    try:
        while True:  # a célula continua rodando: é isso que mantém o link funcionando
            time.sleep(600)
            print(f"   ...no ar há {(time.time() - inicio) / 60:.0f} min")
    except KeyboardInterrupt:
        print("API desligada.")